# Stage A. Do the pretraining corpora differ?

The project holds the model, the objective and the training budget fixed and
changes only the molecules. That rests on a prior question. If the candidate
corpora are chemically the same there is no treatment, and nothing we run
afterwards can measure anything.

**They are not the same.** Four corpora separate on properties, on scaffolds, on
redundancy and on distance to the benchmarks. The largest ordinary-property gap
runs past two standard deviations. So a null result downstream would mean
something rather than nothing.

Two findings cut against the design as written, and both are in here. The
`zinc_nplike` arm does not sit between the other two. And the released
checkpoint has already seen 73 percent of BBBP and 93 percent of ESOL.

The road from here. Sections 1 to 3 say what the corpora are and what the score
the third arm is built on actually tracks. Sections 4 to 7 measure how they
differ and how much they overlap. Sections 8 to 10 bring in the benchmarks.
Section 11 is a picture and section 12 is what the rest of the project gets.

This notebook reads what the pipeline wrote and recomputes nothing expensive, so
it runs in seconds. Build the inputs first, from the repo root:

```
make caches
make stage-a
make reference-distance
```

The third is separate because it is the one step that runs against the whole
pools rather than a 20,000-molecule condition, and it takes about half an hour.
Section 9 is the only part that needs it.

## 0. Setup and provenance

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import Image

from natural_compounds.config import load_config
from natural_compounds.palette import COLOUR, LABEL, ORDER
from natural_compounds.paths import CONFIGS, DATA_PROCESSED, DATA_RAW, RUNS

ANALYSIS = RUNS / "analysis" / "stage_a_corpora"
REPORT = ANALYSIS / "report"
PATHWAY = ANALYSIS / "pathway"
REFERENCE = ANALYSIS / "reference"

# Which command builds what. `make stage-a` deliberately does not build the
# reference distances, so sending a reader there for a missing one wastes an
# afternoon.
BUILDS = {REFERENCE: "make reference-distance"}
DEFAULT_BUILD = "make caches && make stage-a"


def load(name, where=None):
    path = (where or REPORT) / name
    if not path.exists():
        command = BUILDS.get(where or REPORT, DEFAULT_BUILD)
        raise SystemExit(f"{path} is missing.\nBuild it with `{command}` from the repo root.")
    return pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)


SAMPLED = {"zinc_general", "zinc_nplike"}


def by_corpus(frame, column="corpus"):
    return frame.set_index(column).loc[[c for c in ORDER if c in set(frame[column])]].reset_index()


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.precision", 3)

In [ ]:
from natural_compounds.provenance import problems, read_manifests, summary

experiment = load_config(CONFIGS / "experiments" / "stage_a_corpora.yaml")
manifests = read_manifests(ANALYSIS, [".", "report", "pathway", "reference"])
print(pd.DataFrame(summary(manifests)).to_string(index=False))

found = problems(experiment, manifests)
print()
if found:
    for line in found:
        print(f"  ! {line}")
else:
    print("  these outputs match the config and each other")

Four directories, four manifests, and they have to agree with each other and
with the config. Three scripts write them and nothing stops you running one,
editing a YAML, then running the next. The check above is `provenance.py`, with
tests. It catches a changed corpus list, a changed size, a changed task set,
mismatched commits and a different RDKit.

RDKit is the one to watch, because canonical SMILES, scaffolds and fingerprints
are all its output rather than properties of the molecule. The run manifests
record it. The caches do not, since they predate that field and rebuilding them
would rewrite inputs Stage C is using.

## 1. The four corpora

Three arms plus a reference. `zinc_general` is a random draw from ZINC20, a
catalogue of synthesisable compounds. `coconut` is COCONUT, an open database of
natural products. `zinc_nplike` is the tenth of `zinc_general` scoring highest
on natural-product-likeness, which is meant to separate "looks like nature"
from "comes from nature". `pubchem` is CheMeleon's own corpus and is here as a
reference, not as an arm.

In [ ]:
from natural_compounds.cache import cache_path

rows = []
for name in ORDER:
    cfg = load_config(CONFIGS / "corpora" / f"{name}.yaml")
    parent = cfg.get("cache_from", name)
    cache = json.loads(cache_path(parent).with_suffix(".manifest.json").read_text())
    stats = json.loads((DATA_PROCESSED / "corpora" / f"{name}_n20000_s0.stats.json").read_text())
    rows.append(
        {
            "corpus": name,
            "cache": parent,
            "raw rows": cache["raw_rows"],
            "unique": cache["unique"],
            "after cleaning": stats["after_cleaning"],
            "over 150 chars": cache["over_150_chars"],
            "multi-fragment": cache["multifragment"],
            # zinc_nplike keeps a decile of its parent so its pool is not its
            # cleaned cache.
            "pool drawn from": stats.get("after_np_filter", stats["after_cleaning"]),
            "sampled": stats["final"],
            "pool is itself a sample": name in SAMPLED,
        }
    )
pd.DataFrame(rows)

Cleaning follows CheMeleon's rules, the same rules for every corpus, because
cleaning them differently would add a second variable.

Two things the table does not say. The rule is close to CheMeleon's rather than
identical: they drop a molecule when the raw SMILES reaches 150 characters and
we drop when the canonical SMILES exceeds 150, which keeps 4,904 COCONUT
molecules they would drop. We are not changing it, because the draw is a
permutation of the pool and a different pool length gives entirely different
corpora. `docs/open-issues.md` records it.

And the loss is not uniform. Natural products are larger and more often
reported as mixtures, so the rule discards nine percent of COCONUT and three
molecules of PubChem. Cleaning is itself a compositional choice.

## 2. Natural-product-likeness

Ertl's fragment score, as shipped with RDKit. It measures structural
resemblance to natural-product chemistry, not biological origin. Roughly, +1
and above reads as natural-product-like and negative values read as synthetic.

The score is computed per molecule inside `corpus_report.py` alongside the other
descriptors so the means, effect sizes and overlaps elsewhere cover it with no
second code path.

In [ ]:
per_molecule = load("properties_per_molecule.parquet")
frames = {c: per_molecule.xs(c) for c in ORDER}
np_scores = {c: frames[c]["np_likeness"].to_numpy() for c in ORDER}

pd.DataFrame(
    [
        {
            "corpus": c,
            "n": len(v),
            "mean": v.mean(),
            "median": np.median(v),
            "share above 0": (v > 0).mean(),
            "share above +1": (v > 1).mean(),
            "minimum": v.min(),
        }
        for c, v in np_scores.items()
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for c in ORDER:
    ax.hist(np_scores[c], bins=70, histtype="step", density=True, label=LABEL[c], color=COLOUR[c])
ax.axvline(0, color="black", lw=0.6)
ax.axvline(1, color="black", lw=0.6, ls=":")
ax.annotate("0", (0, ax.get_ylim()[1]), fontsize=7, xytext=(2, -10), textcoords="offset points")
ax.annotate(
    "Ertl's +1.0", (1, ax.get_ylim()[1]), fontsize=7, xytext=(3, -10), textcoords="offset points"
)
ax.set_xlabel("natural-product-likeness (Ertl 2008 fragment score, unitless)")
ax.set_ylabel("density")
ax.set_title("Natural-product-likeness, 20,000 molecules per corpus")
ax.legend(fontsize=8)
fig.tight_layout()

The ordering is the one the project assumes and the gap between COCONUT and the
two general libraries is about two units. The contrast Stage D needs is real.

Two things the table says that the project's naming does not. COCONUT is
heterogeneous: a quarter of it scores below zero. We do not filter, because the
score puts caffeine at about -1 and removing molecules on a rule that gets
caffeine wrong would damage the corpus in a direction nobody can state.

And `zinc_nplike` is not natural-product-like in the sense COCONUT is. Its
decile starts near +0.14, under Ertl's +1.0 line, and it is a truncated
distribution where COCONUT is a full one: nothing in it scores below +0.14 by
construction while a quarter of COCONUT does. It is the most
natural-product-like tenth of a synthetic library. Section 12 shows the gap is
wider than this table suggests.

### Is our scorer right?

Almost nothing in Stage A can be checked against a published number. This can.
COCONUT ships its own natural-product-likeness column computed with the same
model so agreeing with it is a check on our wiring rather than a check that the
code does what the code does.

In [ ]:
from natural_compounds.nplikeness import np_likeness

coconut_raw = DATA_RAW / "coconut" / "coconut_csv_lite-11-2026.csv"
if coconut_raw.exists():
    frame = pl.read_csv(
        coconut_raw, columns=["canonical_smiles", "np_likeness"], infer_schema_length=0
    )
    frame = frame.filter(
        pl.col("np_likeness").is_not_null() & (pl.col("np_likeness") != "")
    ).sample(2000, seed=0)
    published = frame["np_likeness"].cast(pl.Float64).to_numpy()
    ours = np_likeness(frame["canonical_smiles"].to_list())
    both = np.isfinite(ours) & np.isfinite(published)
    print(f"  n = {both.sum()}")
    print(
        f"  correlation with COCONUT's published column : {np.corrcoef(ours[both], published[both])[0, 1]:.4f}"
    )
    print(
        f"  mean absolute difference                    : {np.abs(ours[both] - published[both]).mean():.4f}"
    )
    print(
        f"  largest absolute difference                 : {np.abs(ours[both] - published[both]).max():.4f}"
    )
else:
    print("raw COCONUT release not present, skipping")

## 3. What the score tracks inside a corpus

Selecting a corpus on natural-product-likeness also selects it on ordinary
properties. Ertl showed the converse himself: a classifier built from seven
simple descriptors nearly reproduces his fragment score. If that holds in our
corpora then "natural products help" and "heavy, saturated, stereochemically
rich molecules help" are the same sentence until an experiment separates them.

Section 4 argues this from effect sizes between corpora. The direct measurement
is the correlation between the score and each property *inside* one corpus,
where corpus identity is held constant by construction. Spearman, because none
of these are linear.

In [ ]:
from scipy.stats import spearmanr

descriptors = [c for c in per_molecule.columns if c != "np_likeness"]
correlation = pd.DataFrame(
    {
        corpus: [
            spearmanr(frames[corpus]["np_likeness"], frames[corpus][d]).statistic
            for d in descriptors
        ]
        for corpus in ORDER
    },
    index=descriptors,
)
correlation["smallest |r|"] = correlation[ORDER].abs().min(axis=1)
correlation["same sign everywhere"] = np.sign(correlation[ORDER]).nunique(axis=1) == 1
correlation.sort_values("smallest |r|", ascending=False).style.background_gradient(
    cmap="coolwarm", vmin=-1, vmax=1, subset=ORDER
).format("{:.2f}", subset=ORDER + ["smallest |r|"])

Seven descriptors keep their sign in all four corpora. Five of them also hold
|r| above 0.2 everywhere and those five are the robust part of the confound:
nitrogen count, fraction sp3, aromatic rings, aromatic atoms and stereocentres.
The score is anti-nitrogen, anti-aromatic, pro-saturation and
pro-stereochemistry, inside every corpus we hold.

**Oxygen is not part of it.** The literature's signature is oxygen-rich and
nitrogen-poor. Nitrogen-poor holds everywhere, -0.31 to -0.61. Oxygen flips:
+0.23 in COCONUT and +0.18 in PubChem against -0.11 and -0.07 in the two ZINC
arms. Oxygen content is a property of the natural-product databases rather than
of what the score measures. Molecular weight fails the same way and more
clearly: +0.02 inside COCONUT, strongly negative inside ZINC, so **size is not
what the score tracks** either.

One caveat on the `zinc_nplike` column. It is a top-decile cut on the variable
being correlated, so its range is truncated and its correlations are attenuated.
Read its sign, not its magnitude.

The consequence for Stage D: a matched control arm should match on those five
descriptors rather than on size.

## 4. Properties and effect sizes

Seventeen descriptors per molecule, the seven Ertl used among them. Means are
easy to overread so every pair also gets a standardised effect size, a rank
statistic and the overlap of the two histograms.

In [ ]:
properties_table = load("properties.csv")
shown = [
    "mw",
    "logp",
    "tpsa",
    "heavy_atoms",
    "aromatic_rings",
    "fraction_sp3",
    "stereocentres",
    "n_count",
]
properties_table.pivot(index="property", columns="corpus", values="mean").loc[shown, ORDER]

In [ ]:
effects = load("property_effects.csv")
ranked = effects.reindex(effects.cohens_d.abs().sort_values(ascending=False).index)

# The score separates the arms by construction, since zinc_nplike is defined by
# a cut on it. Section 2 covers that. Here the question is the ordinary
# chemistry.
top = ranked.query("property != 'np_likeness'").head(12).reset_index(drop=True)
top[
    [
        "corpus_a",
        "corpus_b",
        "property",
        "mean_a",
        "mean_b",
        "median_a",
        "median_b",
        "cohens_d",
        "cliffs_delta",
        "overlap",
        "tail_driven",
    ]
]

Cohen's d assumes the two spreads are comparable. COCONUT's variance reaches
many times another corpus's on the same property so Cliff's delta sits beside
it: the chance that a random molecule of A exceeds a random molecule of B,
shifted so 0 means no difference. Where the two disagree, believe the rank
statistic.

`tail_driven` marks the rows where they do. Those are differences in one
corpus's tail rather than shifts in the bulk and reading them off the sorted
table turns "COCONUT has a tail of peptides and glycosides" into "COCONUT
molecules are more flexible", which is false of the typical molecule.

In [ ]:
flagged = effects.query("tail_driven")
flagged[
    [
        "corpus_a",
        "corpus_b",
        "property",
        "mean_a",
        "mean_b",
        "median_a",
        "median_b",
        "cohens_d",
        "cliffs_delta",
    ]
]

Read the medians. On rotatable bonds, random ZINC and COCONUT differ by 3.7
bonds on the mean and not at all on the median and d and Cliff's delta point in
opposite directions. On stereocentres, COCONUT and the natural-product-like ZINC
decile have the same median and a Cliff's delta near zero. The typical molecule
of each is indistinguishable on both.

In [ ]:
panels = {
    "mw": "molecular weight (Da)",
    "logp": "logP (Crippen, unitless)",
    "tpsa": "topological polar surface area (A^2)",
    "fraction_sp3": "fraction of sp3 carbons",
    "aromatic_rings": "aromatic rings (count)",
    "stereocentres": "stereocentres (count)",
}
fig, axes = plt.subplots(2, 3, figsize=(13, 6.8))
for ax, (column, label) in zip(axes.ravel(), panels.items(), strict=True):
    for c in ORDER:
        values = frames[c][column].to_numpy()
        upper = np.quantile(values, 0.99)
        ax.hist(
            values[values <= upper],
            bins=50,
            histtype="step",
            density=True,
            label=LABEL[c],
            color=COLOUR[c],
        )
    ax.set_xlabel(label, fontsize=9)
for ax in axes[:, 0]:
    ax.set_ylabel("density")
axes[0, 0].legend(fontsize=7)
fig.suptitle(
    "Property distributions, 20,000 molecules per corpus, each panel trimmed at its 99th percentile"
)
fig.tight_layout()

### How much of this is the draw?

Every number above comes from one sample of twenty thousand per corpus. The
report this notebook accompanies gives no interval on any of them and its own
protocol section asks for paired differences with confidence intervals.

Resampling answers that, but not the same way for every corpus. COCONUT and
PubChem are uniform draws from their pools so resampling molecules is right. The
two ZINC arms come from a cluster sample of 250 of 787 shards with nearly all
the variance between shards, so resampling their molecules would treat 20,000
correlated draws as independent ones. They get their shards resampled instead.

And the two ZINC arms are not two samples, they are two filters over one draw of
shards. A contrast between them is paired at shard level, so the shards are
drawn once and applied to both.

In [ ]:
def clusters_for(corpus):
    """Shard identity for a ZINC arm, None where molecules are independent.

    The provenance file carries source_row, the row in zinc20_spread.csv, and
    that file is written shard by shard at 2,000 rows each.
    """
    if corpus not in SAMPLED:
        return None
    path = DATA_PROCESSED / "corpora" / f"{corpus}_n20000_s0.provenance.parquet"
    provenance = pl.read_parquet(path)
    # The property frame drops anything RDKit cannot parse and the provenance
    # file does not, so the two can fall out of step without saying so. They
    # agree today on all four corpora. Fail rather than silently mislabel.
    assert list(frames[corpus].index) == provenance["smiles"].to_list(), corpus
    return provenance["source_row"].to_numpy() // 2000


def bootstrap_d(x, y, cluster_x=None, cluster_y=None, draws=300, seed=0):
    rng = np.random.default_rng(seed)
    paired = (
        cluster_x is not None
        and cluster_y is not None
        and set(np.unique(cluster_x)) == set(np.unique(cluster_y))
    )
    groups = np.unique(cluster_x) if cluster_x is not None else None

    def draw(values, clusters, picked):
        if clusters is None:
            return rng.choice(values, size=len(values), replace=True)
        return np.concatenate([values[clusters == g] for g in picked])

    out = np.empty(draws)
    for i in range(draws):
        shared = rng.choice(groups, size=len(groups), replace=True) if paired else None
        a = draw(x, cluster_x, shared if paired else _resample(rng, cluster_x))
        b = draw(y, cluster_y, shared if paired else _resample(rng, cluster_y))
        pooled = np.sqrt(
            ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
        )
        out[i] = (a.mean() - b.mean()) / pooled
    return out


def _resample(rng, clusters):
    if clusters is None:
        return None
    groups = np.unique(clusters)
    return rng.choice(groups, size=len(groups), replace=True)


clusters = {c: clusters_for(c) for c in ORDER}
intervals = []
# Four from the top of the table and two involving neither ZINC arm, so the
# cost of the cluster design is visible next to an ordinary interval.
plain = ranked.query("property != 'np_likeness'")
plain = plain[~plain.corpus_a.isin(SAMPLED) & ~plain.corpus_b.isin(SAMPLED)]
chosen = pd.concat([top.head(4), plain.head(2)]).drop_duplicates(
    subset=["corpus_a", "corpus_b", "property"]
)

for _, row in chosen.iterrows():
    a, b, name = row["corpus_a"], row["corpus_b"], row["property"]
    draw = bootstrap_d(
        frames[a][name].to_numpy(dtype=float),
        frames[b][name].to_numpy(dtype=float),
        clusters[a],
        clusters[b],
    )
    lo, hi = np.quantile(draw, [0.025, 0.975])
    intervals.append(
        {
            "comparison": f"{a} vs {b}",
            "property": name,
            "cohens_d": row["cohens_d"],
            "2.5%": lo,
            "97.5%": hi,
            "width": hi - lo,
            "design": "paired shards"
            if a in SAMPLED and b in SAMPLED
            else ("shards" if a in SAMPLED or b in SAMPLED else "molecules"),
        }
    )
pd.DataFrame(intervals)

The clustered intervals run about a quarter to two thirds wider than drawing the
shards once would give. A molecule bootstrap on the ZINC arms would have
reported a narrow number and been wrong. They are still far narrower than the
effects, so the separations here are not an accident of the draw.

That is a narrow claim. It fixes the corpus and asks about the draw. It says
nothing about whether COCONUT is a fair sample of natural-product chemistry or
whether 250 shards represent ZINC20.

The scaffold and sphere counts in the next two sections are a different case
again. Both grow with sample size by construction, so they compare across the
rows of one table and not against a number computed at another size.

## 5. Scaffolds

Bemis-Murcko scaffolds, the ring skeleton with side chains stripped. Acyclic
molecules have an empty scaffold. They are reported as their own share and kept
out of the concentration statistics, because treating the empty string as one
scaffold drops every acyclic molecule into a single bucket and inflates the
concentration of exactly the corpus that has the most of them.

In [ ]:
scaffolds = by_corpus(load("scaffolds.csv"))
scaffolds[
    [
        "corpus",
        "n_ring_molecules",
        "n_scaffolds",
        "scaffolds_per_molecule",
        "normalised_entropy",
        "top100_share",
        "singleton_share_of_scaffolds",
        "empty_scaffold_share",
    ]
]

In [ ]:
Image(filename=str(REPORT / "scaffold_concentration.png"))

PubChem is the most scaffold-concentrated corpus here, not COCONUT. A quarter of
its ring-containing molecules sit on a hundred frameworks. Random ZINC is the
most diverse, which is what an enumerated catalogue should look like.

Two cautions. Denominators differ by 13 percent across rows because the acyclic
share does, and every column grows with the number of molecules measured, so
read the ordering rather than the magnitudes. And `normalised_entropy` divides
by the log of the scaffolds actually seen, which is not the same quantity across
corpora sampled to different depths.

COCONUT's acyclic share is the number that travels downstream and section 10
shows what it does there.

## 6. Redundancy and why nominal size is not effective size

Average pairwise similarity says all four corpora are diverse and nearly
identical to one another. That measure is close to useless here. A mean over
eight million pairs cannot see a few thousand near-duplicates.

Sphere exclusion can. Walk a sample, keep a molecule as a new sphere centre
unless it falls within a set similarity of a centre already kept, then divide
centres by sample size. Two radii, because they answer different questions. At
similarity 0.65 a shared sphere means close analogues. At 0.35, the radius
ChemFM's diversity statistic uses, coverage counts coarse regions of chemical
space.

In [ ]:
diversity = by_corpus(load("diversity.csv"))
shown = pd.DataFrame(
    {
        "corpus": diversity["corpus"].map(LABEL),
        "mean pairwise": diversity["mean_pairwise_tanimoto"].map("{:.3f}".format),
        "pairs above 0.4": [
            f"{p:.3%}  [{lo:.3%}, {hi:.3%}]"
            for p, lo, hi in zip(
                diversity["pairs_over_0.4"],
                diversity["pairs_over_0.4_lo"],
                diversity["pairs_over_0.4_hi"],
                strict=True,
            )
        ],
        "resampled": diversity["resampled"]
        + " ("
        + diversity["resampling_units"].astype(str)
        + ")",
        "near twins at 2,000": (1 - diversity["sphere_coverage_sim0.65_n2000"]).map(
            "{:.1%}".format
        ),
        "near twins at 10,000": (1 - diversity["sphere_coverage_sim0.65_n10000"]).map(
            "{:.1%}".format
        ),
        "coarse coverage at 2,000": diversity["sphere_coverage_sim0.35_n2000"],
        "coarse coverage at 10,000": diversity["sphere_coverage_sim0.35_n10000"],
    }
)
print(shown.to_string(index=False))
print()
print(
    "sphere coverage moves at most "
    f"{diversity[['sphere_order_sd_sim0.65', 'sphere_order_sd_sim0.35']].max().max():.4f}"
    " over twelve shuffles of the same 4,000 molecules"
)

Two statistics agree on the shape. COCONUT's share of pairs above Tanimoto 0.4
is roughly eighty times random ZINC's, while its *mean* similarity is among the
lowest of the four. A heavy tail of close pairs on an otherwise diverse body is
what analogue series look like: the same skeleton decorated many ways, the same
family re-reported from different organisms.

Eighty is the point estimate and the two intervals together allow anything from
forty to a hundred and forty, because the ZINC share rests on a few hundred
pairs out of eight million. The order of magnitude is the result.

Those intervals resample sampling units, molecules for COCONUT and PubChem and
shards for the two ZINC arms. Pairs where a unit met its own duplicate are
dropped: a molecule matches its copy at 1.0, and on random ZINC those fake pairs
outnumber the real ones by 2.6 to one, which would push the whole interval above
the point it is meant to bracket.

So at a fixed count of molecules COCONUT carries fewer distinct coarse regions
and several times PubChem's share of near twins.

Two limits travel with this. Greedy sphere exclusion depends on the order it
walks the sample, so the line under the table is the spread over twelve
shuffles: 0.003 at most, against a spread between corpora of 0.27. And the four
samples are drawn at very different rates from their parents, so two members of
one analogue series can hardly both land in a sparse draw. These numbers
describe redundancy within a draw of this size, which is what a pretraining arm
would see and not the analogue structure of ZINC20 itself.

## 7. How much the corpora overlap

Two questions that get confused. Do the corpora share frameworks, and do their
molecules sit near one another?

Both sides of every number in this section are 4,000-molecule draws from the
20,000-molecule conditions. Nearest-neighbour similarity rises with the size of
the reference set, so these figures compare across the rows of their own table
and not with section 9, which measures against whole pools.

In [ ]:
jaccard = (
    load("scaffold_overlap.csv", where=ANALYSIS)
    .set_index("Unnamed: 0")
    .rename_axis(None)
    .loc[ORDER, ORDER]
)
weighted = (
    load("scaffold_overlap_weighted.csv", where=ANALYSIS)
    .set_index("Unnamed: 0")
    .rename_axis(None)
    .loc[ORDER, ORDER]
)
print("Jaccard of scaffold SETS")
print(jaccard.round(4).to_string())
print()
print("share of the ROW corpus's molecules whose scaffold the COLUMN corpus also has")
print(weighted.round(3).to_string())

The first table says every pair is disjoint and tells us nothing. The reason is
in section 5: between 82 and 93 percent of the scaffolds in these corpora occur
once, so a set Jaccard is mostly a count of rare-framework coincidences.

The second weights by molecules and discriminates. Read it along a row. Of
COCONUT's molecules, 27 percent sit on a framework PubChem also holds and 11
percent on one random ZINC also holds. The relation is asymmetric because
coverage is, and that is the quantity transfer would care about.

In [ ]:
neighbours = load("nearest_neighbour.csv")
neighbours.pivot(index="from_corpus", columns="to_corpus", values="mean_nn_similarity").loc[
    ORDER, ORDER
]

In [ ]:
pair = neighbours.set_index(["from_corpus", "to_corpus"])
for a, b in [("coconut", "pubchem"), ("coconut", "zinc_general")]:
    there, back = pair.loc[(a, b), "share_over_0.4"], pair.loc[(b, a), "share_over_0.4"]
    print(f"{a:13} -> {b:13} {there:6.1%}    {b:13} -> {a:13} {back:6.1%}")

The asymmetry is the useful result. COCONUT sits inside PubChem's coverage
considerably more than PubChem sits inside COCONUT's. PubChem is assembled from
the published literature and already holds much natural-product chemistry so
COCONUT is closer to a dense subregion of PubChem than to an independent region
of chemical space. ZINC is genuinely elsewhere.

One row of this table is not chemistry. `zinc_general` and `zinc_nplike` are
drawn from the same cache and the decile is a subset so the two samples share
molecules that match themselves at similarity 1.0. Read that pair as nesting,
not as proximity.

## 8. What the corpora already contain of the benchmarks

Exposure is how many benchmark molecules a corpus already holds. It is a
contamination question and it is asked of the whole pool, not of a condition.

In [ ]:
full = by_corpus(load("full_corpus_exposure.csv"))
view = full.pivot(index="corpus", columns="task", values=["exact_share", "exact_share_cleaned"])
print(view.loc[[c for c in ORDER if c in view.index]].to_string(float_format=lambda x: f"{x:.1%}"))
print()
print(
    full[
        [
            "corpus",
            "pool_size",
            "cleaned_pool_size",
            "pool_is_sample",
            "task",
            "expected_in_50k_this_task",
            "expected_in_50k_any_task",
        ]
    ].to_string(index=False)
)

Two pools, because they answer different questions. The raw cache says how much
of the benchmark is in the source at all. The cleaned pool says how much could
reach an arm, and it is the one a size-matched comparison needs: cleaning costs
COCONUT nine percent and PubChem nothing. At a matched 50,000, PubChem carries
about 127 already-seen structures, COCONUT 92, and both ZINC arms zero.

`pool_is_sample` marks the rows where the pool is a draw rather than a database.
Both ZINC arms read 500,000 molecules sampled from a 1.5-billion-row upload, so
the zero means our arm contains none, not that ZINC20 contains none.

The PubChem row says something about the corpus rather than the benchmark.
CheMeleon's paper calls its pretraining set a random million from PubChem, and
PubChem holds around 119 million compounds, so a uniform draw would contain well
under one percent of any benchmark rather than seventy or ninety. The released
file is the well-characterised core, enriched by about two orders of magnitude.
That is a property of the released checkpoint too.

None of this is label leakage. Pretraining is unsupervised descriptor regression
and never sees a benchmark label. `docs/protocol.md` keeps the three apart.

### Which molecules, exactly

The counts above are abstract. These are the structures. Seeing them makes the
confound concrete and it is a check on the matching code that no table can give.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

from natural_compounds.chem import canonicalize
from natural_compounds.paths import resolve_source

RDLogger.DisableLog("rdApp.*")

task_cfg = load_config(CONFIGS / "tasks" / "bbbp.yaml")
bbbp = pd.read_csv(resolve_source(task_cfg["source"]))
bbbp_canonical = {
    c for c in (canonicalize(s) for s in bbbp[task_cfg["smiles_column"]].astype(str)) if c
}

condition = pd.read_csv(DATA_PROCESSED / "corpora" / "pubchem_n20000_s0.csv")["smiles"].tolist()
inside = sorted(bbbp_canonical & set(condition))
print(f"{len(inside)} BBBP structures sit inside the 20,000-molecule PubChem condition")

Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in inside[:12]], molsPerRow=4, subImgSize=(230, 180)
)

## 9. How far the benchmarks sit from each corpus

Proximity is how close a benchmark molecule sits whether or not the corpus
contains it. Two references answer two different questions and the section does
both, because they differ by more than a caveat can carry.

In [ ]:
proximity = load("task_proximity.csv")
table = proximity.pivot(index="corpus", columns="task", values="mean_nn_similarity")
table.loc[ORDER].rename(index=LABEL)

In [ ]:
per_mol = load("task_proximity_per_molecule.csv")
names = {"bbbp": "BBBP, blood-brain barrier", "esol": "ESOL, aqueous solubility"}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9), sharey=True)
for ax, task in zip(axes, sorted(per_mol.task.unique()), strict=True):
    for c in ORDER:
        ax.hist(
            per_mol.query("task == @task and corpus == @c").nearest_similarity,
            bins=40,
            histtype="step",
            density=True,
            label=LABEL[c],
            color=COLOUR[c],
        )
    ax.set_title(
        f"{names.get(task, task)}  (n={per_mol.query('task == @task').row_index.nunique()})"
    )
    ax.set_xlabel("Tanimoto to the nearest molecule of the 20,000-molecule condition")
    ax.set_xlim(0, 1)
axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)
fig.suptitle("Benchmark molecules against a condition of the size Stage D would run")
fig.tight_layout()

Against a condition of this size, PubChem is nearest to both benchmarks on every
statistic and about half of each benchmark has no neighbour above Tanimoto 0.3
in a ZINC condition. That is the ordering to expect downstream if transfer is
driven by coverage rather than by anything specific to natural products.

This is the right reference for Stage D, where an arm really is 20,000 or 50,000
molecules. It is the wrong one for Stage B.

### Against the corpus the checkpoint actually saw

Stage B fine-tunes the released CheMeleon checkpoint, and that checkpoint saw
all 944,286 molecules of `cleaned_pubchem_1MM.smiles`. Nearest-neighbour
similarity rises with the size of the reference, so the condition figure and the
pool figure are different quantities and the gap is not a constant.

In [ ]:
gap = load("reference_distance_gap.csv", where=REFERENCE)
print(
    gap[
        [
            "task",
            "corpus",
            "reference_size_condition",
            "mean_nn_similarity_condition",
            "reference_size_full",
            "mean_nn_similarity_full",
            "understated_by",
        ]
    ].to_string(index=False)
)

The gap runs from 0.03 for `zinc_nplike` to 0.43 for PubChem. It depends on how
dense the corpus is near a given molecule, so no single correction recovers one
number from the other.

What that does to Stage B is the point.

In [ ]:
distances = load("reference_distance_per_molecule.csv", where=REFERENCE)
for task in ["bbbp", "esol"]:
    rows = distances.query("task == @task and corpus == 'pubchem'").nearest_similarity_full
    print(f"{task}: {len(rows)} molecules against the whole 944,286")
    for cut in (0.5, 0.7, 0.9):
        print(f"   below {cut}: {(rows < cut).sum():4d}  ({(rows < cut).mean():.1%})")

**There is almost no far bin.** Against the corpus the checkpoint was really
trained on, 68 of ESOL's 1,128 molecules sit below similarity 0.9 and 15 below
0.5. BBBP is thin rather than empty, 303 and 64 of 2,039.

So "bin the test molecules by distance to the pretraining corpus and ask where
the gain concentrates" is not answerable on ESOL as it stands, and on BBBP only
in a tail of a few dozen molecules. That is a Stage A finding about a Stage B
design, and it is better known now than after the fine-tuning runs.
`docs/open-issues.md` lists the three ways out and recommends the cheapest,
which is to report Stage B as a statement about the near regime and say so.

## 10. What the frozen splits never test

Stage A hands the frozen splits to every later stage, so what they contain is a
Stage A result too.

In [ ]:
from natural_compounds.chem import murcko_scaffold

for task in ["esol", "bbbp"]:
    cfg = load_config(CONFIGS / "tasks" / f"{task}.yaml")
    frame = pd.read_csv(resolve_source(cfg["source"]))
    scaffold = [murcko_scaffold(s) for s in frame[cfg["smiles_column"]].astype(str)]
    acyclic = {i for i, s in enumerate(scaffold) if s == ""}
    print(f"{task}: {len(frame)} rows, {len(acyclic)} acyclic ({len(acyclic) / len(frame):.0%})")
    for seed in range(5):
        split = json.loads((DATA_PROCESSED / "splits" / task / f"seed{seed}.json").read_text())
        counts = {k: len(acyclic & set(split[k])) for k in ("train", "val", "test")}
        print(
            f"   seed{seed}  acyclic in train {counts['train']:>4}   val {counts['val']:>4}   test {counts['test']:>4}"
        )

In [ ]:
cfg = load_config(CONFIGS / "tasks" / "esol.yaml")
frame = pd.read_csv(resolve_source(cfg["source"]))
y = frame[cfg["target_columns"][0]].to_numpy(float)
acyclic = np.array([murcko_scaffold(s) == "" for s in frame[cfg["smiles_column"]].astype(str)])
print(f"  acyclic  n={acyclic.sum():4d}  mean {y[acyclic].mean():+.2f}")
print(f"  cyclic   n={(~acyclic).sum():4d}  mean {y[~acyclic].mean():+.2f}")
print(
    f"  gap {abs(y[acyclic].mean() - y[~acyclic].mean()):.2f} log units, {abs(y[acyclic].mean() - y[~acyclic].mean()) / y.std(ddof=1):.2f} standard deviations"
)
means = [
    y[json.loads((DATA_PROCESSED / "splits" / "esol" / f"seed{s}.json").read_text())["test"]].mean()
    for s in range(5)
]
print(
    f"  test mean across seeds: {min(means):+.2f} to {max(means):+.2f}, a swing of {max(means) - min(means):.3f}"
)

317 of ESOL's 1,128 molecules have no ring, so they share the empty scaffold and
move as one group. That group is larger than half the test subset, so the
balanced scaffold rule places it first and it fits in train every time.
**Zero acyclic molecules reach the ESOL test set in any of the five seeds.**
BBBP behaves the same way with its 99.

So it is not variance, it is a constant shift. Acyclic ESOL molecules are nearly
a standard deviation more soluble than the rest, identically in every seed.
Twenty-eight percent of ESOL's chemistry is never tested.

A paired comparison between corpora survives this, since every arm sees the same
split. What does not survive is calling an ESOL result a statement about ESOL.
It is a statement about ring-containing ESOL, and the 0.94 log unit swing
between seeds sits on top of that.

## 11. The map

A t-SNE of Morgan fingerprints, corpora and benchmark molecules together.

Read it with care and read the tables first. t-SNE preserves neighbourhoods and
not distances. The apparent size of a cluster means nothing and the gaps between
clusters mean nothing either. It is here because it is legible, not because it
is evidence. Every claim this notebook makes rests on the numbers in sections 2
to 10.

In [ ]:
Image(filename=str(ANALYSIS / "tsne_corpora.png"))

## 12. What Stage A establishes

**The treatment is real.** Four corpora separated on properties, on scaffolds,
on redundancy and on distance to the benchmarks, with the largest
ordinary-property gap past two standard deviations.

So a null downstream would mean something, with one condition: it would mean the
model was indifferent to a real difference, but only where the instrument can
resolve an effect. Section 10 shows ESOL cannot. Its test mean swings 0.94 log
units across the frozen seeds and it never scores an acyclic molecule, so there
"no effect" and "no instrument" are not separable.

Four variables move together across the Stage D arms and no size-matched design
separates them on its own.

| | NP-likeness | property profile | exposure per 50k | proximity |
|---|---|---|---|---|
| PubChem | low | flat, aromatic, few stereocentres | 127 | nearest |
| COCONUT | high | heavy, oxygen-rich, stereo-rich | 92 | middle |
| ZINC, random | lowest | drug-like, synthetic | 0 | far |
| ZINC, NP-like decile | middle | small, saturated, stereo-rich | 0 | farthest |

Reading a Stage D difference as "natural products help" means ruling out the
other columns. Section 3 narrows the property column to the five descriptors a
matched control arm would have to match, and size is not among them.

**`zinc_nplike` is the arm to be careful with.** It was meant to sit between the
control and the treatment so that a result could separate "looks like nature"
from "comes from nature". It does not: it is further from COCONUT than random
ZINC on 13 descriptors of 17, mean |d| 0.80 against 0.60. A Stage D result where
it scores below `zinc_general` is consistent with the arm simply being further
away. That also contradicts the plan's compute-shortage rule, which drops
`zinc_general` first because `zinc_nplike` supposedly carries the contrast. Drop
`zinc_nplike` first instead.

What the rest of the project gets:

- **Stage B**: the per-molecule distances of section 9, keyed to the row index
  the frozen splits use, against the whole pool as well as the condition, and
  the finding that the near regime is almost all there is.
- **Stage C**: nominal size and effective size diverge by corpus, which argues
  for keeping the size curve on one corpus.
- **Stage D**: the table above and the specification for a matched control arm.
- **Stage E**: the same per-molecule distances, as its binning variable.

Three things Stage A does **not** establish. It does not show that natural
products carry transferable structure, since no measurement here involves a
model that transfers anything. It does not show that COCONUT would pretrain
better, since diversity and distance are not evidence about transfer. And it
says nothing about whether 250 shards represent ZINC20 or whether COCONUT
represents natural-product chemistry, because no resampling of our own files can
reach a question about the database it came from.

What is still open is in `docs/open-issues.md`.

## Appendix A. Do the two scripts agree?

`analyze_corpora.py` and `corpus_report.py` compute corpus-to-corpus similarity
on their own separate samples. Be precise about what that buys: the two share
`fingerprint_matrix` and `tanimoto_matrix`, so a fault in the representation or
in the similarity would appear identically in both. What differs is the draw.
This is a check that the quantity is stable across samples, not that two
implementations agree.

In [ ]:
older = load("corpus_similarity.csv", where=ANALYSIS).set_index("Unnamed: 0").rename_axis(None)
newer = neighbours.pivot(index="from_corpus", columns="to_corpus", values="mean_nn_similarity")

check = pd.DataFrame(
    [
        {"from": a, "to": b, "analyze_corpora": older.loc[a, b], "corpus_report": newer.loc[a, b]}
        for a in ORDER
        for b in ORDER
        if a != b
    ]
)
check["difference"] = check.analyze_corpora - check.corpus_report
print(f"largest disagreement  {check.difference.abs().max():.4f} Tanimoto")
print(
    f"rank agreement        {spearmanr(check.analyze_corpora, check.corpus_report).statistic:.3f} over {len(check)} ordered pairs"
)
check.sort_values("corpus_report", ascending=False)

Eleven of the twelve ordered pairs agree to within 0.004 Tanimoto, which is
well inside what two independent 4,000-molecule draws should differ by. The
ranking survives.

The twelfth is `coconut` against `zinc_nplike`, 0.292 against 0.266, and we do
not have an explanation for it. It is one-sided: the reverse direction moves
0.002. An earlier version of this note blamed the nesting between the two ZINC
arms, which is wrong, since those two pairs are among the closest in the table.

Two checks in this notebook do reach outside the code. The
natural-product-likeness comparison in section 2 matches a column COCONUT
publishes, at r = 1.0000. The similarity code was checked against RDKit's own
`BulkTanimotoSimilarity`, largest difference 1.5e-08. Those are the only two.

## Appendix B. NPClassifier's class structure

Written as the project's own test of its premise. It cannot be that.

COCONUT carries an `np_classifier_pathway` for 92.9 percent of its molecules and
those labels are not observed biology. NPClassifier is a feed-forward network
over counted Morgan fingerprints, and COCONUT fills the column by sending every
structure to its public API. Recovering the label from structure is therefore
one model reproducing another in nearly the same representation: on a
scaffold-held-out split a logistic regression scores 0.829 balanced accuracy
against a floor of 1/7, and a plain nearest-neighbour lookup scores 0.814 with
no model at all.

Two further defects, both measured here. The column is a single-label projection
of a multi-label model with an alphabetical tie-break, and 3 of 40 molecules
queried against the live API come back with two pathways. And the 52,268 rows
with no pathway score +0.57 on natural-product-likeness against +1.20 for the
rest, so dropping them conditions the reference on the model's own confidence.

What survives is geometric: for each molecule, how much its five nearest COCONUT
neighbours agree on class. That asks where a corpus sits relative to the interior
of those classes and does not need the labels to be biological.

In [ ]:
concordance = load("concordance.csv", where=PATHWAY)
bands = [c for c in concordance.columns if c.startswith("band_")]
# The band columns are means and the n columns say how many molecules each
# rests on. The widest band holds only a few dozen ZINC molecules.
interleaved = [c for band in bands for c in (band, band.replace("band_", "n_"))]
concordance[["corpus", "mean_concordance", "median_nn_similarity", *interleaved]]

In [ ]:
Image(filename=str(PATHWAY / "pathway_concordance.png"))

Held-out COCONUT is the ceiling and shuffled labels the floor, both drawn.
Above similarity 0.40 only a few dozen ZINC molecules land at all, so those
points carry their counts and should be read as noise.

The bands are a coarse control, so the table below closes them: each corpus is
resampled in bins of 0.01 until its similarity distribution matches
`zinc_nplike` exactly. Most of COCONUT's apparent lead is distance, +0.210 raw
falling to +0.081 matched. The gap to random ZINC barely moves, +0.076 to
+0.073.

So the surviving result is narrow. **`zinc_nplike` sits nearer the boundaries of
NPClassifier's classes than plain random ZINC does, at matched distance from
COCONUT.** It is not a statement about biosynthesis and a ZINC molecule has
none. Section 3 and section 12 make the same point about that arm by a route
that owes nothing to NPClassifier, which is why they and not this appendix carry
it.

In [ ]:
matched = load("concordance_matched.csv", where=PATHWAY)
print("concordance over zinc_nplike, before and after matching the similarity distributions")
print(
    matched[
        [
            "corpus_a",
            "raw_gap",
            "matched_gap",
            "mean_similarity_a",
            "mean_similarity_b",
            "n_matched",
        ]
    ].to_string(index=False)
)
print()


def distance_to_coconut(other):
    rows = effects[
        ((effects.corpus_a == "coconut") & (effects.corpus_b == other))
        | ((effects.corpus_b == "coconut") & (effects.corpus_a == other))
    ]
    return rows.set_index("property").cohens_d.abs()


against = pd.DataFrame(
    {
        "zinc_nplike": distance_to_coconut("zinc_nplike"),
        "zinc_general": distance_to_coconut("zinc_general"),
    }
)
against["nplike further"] = against.zinc_nplike > against.zinc_general
print(
    f"zinc_nplike is further from COCONUT on {against['nplike further'].sum()} of {len(against)} descriptors"
)
print(
    f"mean |d| to COCONUT:  zinc_nplike {against.zinc_nplike.mean():.3f}   zinc_general {against.zinc_general.mean():.3f}"
)
against.sort_values("zinc_nplike", ascending=False)

The four descriptors where the Ertl filter does move ZINC toward COCONUT are the
score itself, stereocentres, nitrogen count and hydrogen-bond donors. On size,
polarity and oxygen content it moves away, and those carry the larger effects.
The cut buys part of the signature and pays for it in everything else.